# Neuro-tamagotchi

Demo of the `hotb_2026.tamagotchi` package: a pet whose happiness follows your
EEG `UserState` (`RELAX` / `FOCUS` / `OVERLOAD`).

Everything runs against a temporary directory, so the notebook is safe to
execute repeatedly — no global pet state is touched.

In [ ]:
import tempfile
from pathlib import Path

import hotb_2026.tamagotchi as tmg
from hotb_2026.tamagotchi import UserState

tmp = tempfile.mkdtemp(prefix="hotb_tamagotchi_")
import os
os.environ["HOTB_TAMAGOTCHI_HOME"] = tmp
print("state dir:", tmp)

## 1. Create a pet

A new pet starts as an `EGG` with mid-range stats.

In [ ]:
pet = tmg.new_pet("Brainy", "slime")
pet.save()
print(tmg.build_frame(pet))

## 2. The only source of truth: `HAPPINESS_EFFECT`

Happiness changes **per reported state** (not per tick): `RELAX: -1`,
`FOCUS: +10`, `OVERLOAD: -9`.

In [ ]:
from hotb_2026.tamagotchi import HAPPINESS_EFFECT

for state, effect in HAPPINESS_EFFECT.items():
    print(f"{state.value:8} -> happiness {effect:+.0f}")

## 3. Simulating the classifier

A mock stand-in for `hotb_2026.classification`: it emits a stream of
states as the EEG loop would after each window. Each reported state is
pushed into the engine with `apply_state`, then the frame is re-rendered.

In [ ]:
class MockStateClassifier:
    """Stands in for the real EEG classifier."""

    def __init__(self, states):
        self.states = list(states)

    def stream(self):
        yield from self.states


mock = MockStateClassifier([
    UserState.FOCUS,
    UserState.FOCUS,
    UserState.RELAX,
    UserState.OVERLOAD,
    UserState.FOCUS,
    UserState.OVERLOAD,
    UserState.OVERLOAD,
])

for state in mock.stream():
    tmg.apply_state(pet, state)
    mood = tmg.resolve_mood(pet)
    print(f"{state.value:8} -> happiness {pet.stats.happiness:5.1f}  [{mood.value}]")

In [ ]:
print(tmg.build_frame(pet))

## 4. Time-based decay (downtime is not free)

`decay` works on wall-clock timestamps with per-hour rates from the pet
plugin params. A single call after a long pause applies the full
accumulated drop — a daemon outage does not freeze the pet. Happiness is
never time-decayed; only `apply_state` moves it.

In [ ]:
# Pretend 3 hours passed since the last update.
pet.last_update -= 3 * 3600
tmg.decay(pet)
print(tmg.build_frame(pet))

## 5. Evolution: time AND average

`check_evolution` promotes the pet when it has spent enough time in the
current stage *and* its average stats clear the plugin threshold.

In [ ]:
# Force high stats and rewind the stage clock to trigger evolution.
from hotb_2026.tamagotchi.pet import Stats, Stage

for stat in ("happiness", "hunger", "energy", "hygiene"):
    setattr(pet.stats, stat, 90.0)
pet.stage_started_at -= 10_000  # slime: BABY needs >= 300 s and avg >= 35

tmg.check_evolution(pet)
print("stage:", pet.stage)
print(tmg.build_frame(pet))

## 6. Feeding states from a real pipeline

In production the loop looks like this (pseudocode for the streamer side):

```python
pet = tmg.Pet.load()
for window in eeg_streamer:
    state = classifier.predict(window)          # "FOCUS" / "RELAX" / "OVERLOAD"
    tmg.apply_state(pet, UserState.from_value(state))
    pet.save()                                   # daemon renders status.txt
```

The daemon (`python -m hotb_2026.tamagotchi daemon`) independently
applies decay, resolves mood, evolves and renders `status.txt` every
tick (default 60 s). The shell hook (`install-hook`) prints the frame
before each prompt.

## 7. Cleaning up

In [ ]:
import shutil
shutil.rmtree(tmp)